## Importing Libraries

In [ ]:
# 1. SQL finds exact matches (=, >, <), but vector search needs the "closest" match,
#    so the distance has to be calculated for every row.
# 2. Normal indexes (B-tree, hash) work on a single value, not on a vector of 3072 numbers,
#    so every query ends up scanning the whole table.

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from dotenv import load_dotenv

## Loading Environment Variables

In [2]:
load_dotenv()  # Load environment variables from .env file

True

## Creating Qdrant Client and Collection

In [3]:
qdrantclient = QdrantClient(path="qdrant_data")

if not qdrantclient.collection_exists("ipl_players"):
    qdrantclient.create_collection(
        collection_name="ipl_players",
        vectors_config={"size": 3072, "distance": "Cosine"},
    )

## Creating Documents

In [4]:
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


## Creating Embedding Model

In [5]:
#Creating the Embedding Model Object
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

## Creating Vector Store

In [6]:
vector_store = QdrantVectorStore(
    embedding=embeddings,
    collection_name="ipl_players",
    client=qdrantclient
)

## Adding Documents to Vector Store

In [7]:
docs=vector_store.add_documents([doc1, doc2, doc3, doc4, doc5])
print(docs)

['505e427077a14eb5b0e8258fb142302c', '07a0c124f97342a3b57992a9ab81d88a', '22b8db5276c54f3792bf5b051c80ed60', 'b45a9152b23e47e1ac52bb7f0ddd7b4b', 'b8bf9bb861b84051999c0bc7698cab2f']


## Fetching Documents using Scroll

In [8]:
point,offset=qdrantclient.scroll(
    collection_name="ipl_players",
    with_payload=True,
    with_vectors=True,
    offset=None,
    limit=3
)

## Printing Documents, Metadata and Vectors

In [9]:
print("total embedding vectors in vector store : ",len(point))
print("\n\nAll Documents : ", point)
print("\n\n document 1 content : ",point[0].payload["page_content"])
print("\n\n document 1 Meta Data : ",point[0].payload["metadata"])
print("\n\n document 1 embeding vector  : ",point[0].vector)

total embedding vectors in vector store :  3


All Documents :  [Record(id='07a0c124f97342a3b57992a9ab81d88a', payload={'page_content': "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.", 'metadata': {'team': 'Mumbai Indians'}}, vector=[-0.019897207617759705, 0.010927136056125164, 0.017306027933955193, -0.06748199462890625, 0.0007479704800061882, 0.006261706817895174, -0.005775259807705879, 0.024282710626721382, -0.02894914150238037, 0.008916965685784817, -0.006516733672469854, -0.011961684562265873, -0.0010198550298810005, 0.045960914343595505, 0.11539202928543091, -0.0033122384920716286, -0.01577840745449066, 0.0024774547200649977, 0.012389501556754112, -0.001050180639140308, 0.0057808696292340755, -0.005331092048436403, 0.008732625283300877, -0.01676148548722267, -0.015429094433784485, -0.011146051809191704, 0.007015688810497522, -0.009453313425183296, 0

## Accessing a Specific Document

In [10]:
# point ids returned by add_documents
doc_ids = docs
print(doc_ids)

target_id = doc_ids[0]

['505e427077a14eb5b0e8258fb142302c', '07a0c124f97342a3b57992a9ab81d88a', '22b8db5276c54f3792bf5b051c80ed60', 'b45a9152b23e47e1ac52bb7f0ddd7b4b', 'b8bf9bb861b84051999c0bc7698cab2f']


### Get document by id using LangChain

In [13]:
result = vector_store.get_by_ids([target_id])
print(result[0].page_content)
print(result[0].metadata)


Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
{'team': 'Royal Challengers Bangalore', '_id': '505e427077a14eb5b0e8258fb142302c', '_collection_name': 'ipl_players'}


### Get document by id using Qdrant client

In [14]:
points = qdrantclient.retrieve(
    collection_name="ipl_players",
    ids=[target_id],
    with_payload=True,
    with_vectors=True,
)
p = points[0]
print("ID       :", p.id)
print("Content  :", p.payload["page_content"])
print("Metadata :", p.payload["metadata"])
print("Vector   :", p.vector[:5], "... length =", len(p.vector))

ID       : 505e427077a14eb5b0e8258fb142302c
Content  : Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
Metadata : {'team': 'Royal Challengers Bangalore'}
Vector   : [-0.0098205441609025, 0.025457629933953285, 0.024027816951274872, -0.0640832707285881, -0.003316807560622692] ... length = 3072


### Get documents using metadata filter

In [15]:
from qdrant_client import models

mi_points, _ = qdrantclient.scroll(
    collection_name="ipl_players",
    scroll_filter=models.Filter(
        must=[models.FieldCondition(
            key="metadata.team",
            match=models.MatchValue(value="Mumbai Indians"),
        )]
    ),
    limit=10,
    with_payload=True,
    with_vectors=False,
)
for pt in mi_points:
    print(pt.id, "->", pt.payload["page_content"][:60])

07a0c124f97342a3b57992a9ab81d88a -> Rohit Sharma is the most successful captain in IPL history, 
1379a62e0a194d3590a91e01f72c57ef -> Rohit Sharma is the most successful captain in IPL history, 
b45a9152b23e47e1ac52bb7f0ddd7b4b -> Jasprit Bumrah is considered one of the best fast bowlers in
dc5d14403fcf450d9b24f1a3757220b9 -> Jasprit Bumrah is considered one of the best fast bowlers in


## Updating a Document

### Update metadata only

In [16]:
qdrantclient.set_payload(
    collection_name="ipl_players",
    payload={"metadata": {"team": "Royal Challengers Bengaluru", "role": "Batsman"}},
    points=[target_id],
)

updated = qdrantclient.retrieve("ipl_players", ids=[target_id], with_payload=True)
print(updated[0].payload["metadata"])

{'team': 'Royal Challengers Bengaluru', 'role': 'Batsman'}


### Update page content
The embedding has to be created again for new text, so the document is added again with the same id. Qdrant replaces the old point.

In [17]:
new_doc = Document(
    page_content="Virat Kohli is one of the greatest batsmen in IPL history and the all-time leading run scorer. Known for his aggressive batting and fitness.",
    metadata={"team": "Royal Challengers Bengaluru", "role": "Batsman"},
)

vector_store.add_documents([new_doc], ids=[target_id])

updated = vector_store.get_by_ids([target_id])
print(updated[0].page_content)
print(updated[0].metadata)

Virat Kohli is one of the greatest batsmen in IPL history and the all-time leading run scorer. Known for his aggressive batting and fitness.
{'team': 'Royal Challengers Bengaluru', 'role': 'Batsman', '_id': '505e427077a14eb5b0e8258fb142302c', '_collection_name': 'ipl_players'}


## Deleting a Document

### Delete by id

In [18]:
delete_id = doc_ids[4]

vector_store.delete(ids=[delete_id])

print(vector_store.get_by_ids([delete_id]))

[]


### Delete using metadata filter

In [19]:
qdrantclient.delete(
    collection_name="ipl_players",
    points_selector=models.FilterSelector(
        filter=models.Filter(
            must=[models.FieldCondition(
                key="metadata.team",
                match=models.MatchValue(value="Chennai Super Kings"),
            )]
        )
    ),
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

### Check remaining documents

In [20]:
remaining, _ = qdrantclient.scroll("ipl_players", limit=10, with_payload=True)
print("Documents left :", len(remaining))
for r in remaining:
    print(r.id, "->", r.payload["page_content"][:50])

Documents left : 6
07a0c124f97342a3b57992a9ab81d88a -> Rohit Sharma is the most successful captain in IPL
1379a62e0a194d3590a91e01f72c57ef -> Rohit Sharma is the most successful captain in IPL
505e427077a14eb5b0e8258fb142302c -> Virat Kohli is one of the greatest batsmen in IPL 
67e3f528522f4818af5f45b0017fb180 -> Virat Kohli is one of the most successful and cons
b45a9152b23e47e1ac52bb7f0ddd7b4b -> Jasprit Bumrah is considered one of the best fast 
dc5d14403fcf450d9b24f1a3757220b9 -> Jasprit Bumrah is considered one of the best fast 


In [23]:
#Checking semantic similarity


res=vector_store.similarity_search(
    query="which player is the bowler ?",
    k=2
)
print("length : ",len(res))
print("\n\n",res)

length :  2


 [Document(metadata={'team': 'Mumbai Indians', '_id': 'dc5d14403fcf450d9b24f1a3757220b9', '_collection_name': 'ipl_players'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'), Document(metadata={'team': 'Mumbai Indians', '_id': 'b45a9152b23e47e1ac52bb7f0ddd7b4b', '_collection_name': 'ipl_players'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]


In [25]:
res=vector_store.similarity_search_with_score(
    query="which player is the bowler ?",
    k=2
)
doc,score=res[0]

print("Document : ",doc)
print("\n\nScore  : ",score)


Document :  page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.' metadata={'team': 'Mumbai Indians', '_id': 'dc5d14403fcf450d9b24f1a3757220b9', '_collection_name': 'ipl_players'}


Score  :  0.6873008391142263


In [29]:
from qdrant_client import models

res = vector_store.similarity_search_with_score(
    query="player",   # any non-empty text
    k=5,
    filter=models.Filter(
        must=[models.FieldCondition(
            key="metadata.team",
            match=models.MatchValue(value="Mumbai Indians"),
        )]
    ),
)

for doc, score in res:
    print(score, "->", doc.page_content[:60])

0.6091984291143137 -> Rohit Sharma is the most successful captain in IPL history, 
0.6091984285814844 -> Rohit Sharma is the most successful captain in IPL history, 
0.6044482183969285 -> Jasprit Bumrah is considered one of the best fast bowlers in
0.6044482183063544 -> Jasprit Bumrah is considered one of the best fast bowlers in
